# Example of evaluating electron repulsion integrals (ERIs) of Gausslet orbitals

In [1]:
# see the README for build and install instructions of this package
import gausslectronitegrate as gli

In [2]:
# Gausslet G10 coefficients
import sys
sys.path.append("../gausslets/")
from gausslet_coefficients import get_gausslet_g10_coeffs
b_list, _ = get_gausslet_g10_coeffs()

In [3]:
# truncation tolerance
# (values close to zero imply higher accuracy but longer computation time)
tol = 1e-16

In [4]:
# compute the overlap integral for all Gausslets centered at the origin
gli.compute_eri_gausslet_integral(b_list, ([0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]), tol)

1.6252009000830485

In [5]:
# evaluate the overlap integral for four other points, as an example
point_a = [ 1,  0,  0]
point_b = [ 0,  0, -1]
point_c = [ 0,  0,  1]
point_d = [ 0,  0,  0]
gli.compute_eri_gausslet_integral(b_list, (point_a, point_b, point_c, point_d), tol)

-0.0005864045861050048

In [6]:
# define a symmetric Cartesian grid by tuples of the form (istart, num points) in each coordinate direction
grid = [
    (-1, 3),  # x
    (-1, 3),  # y
    (-1, 3),  # z
]

In [7]:
# overall number of grid points
gli.grid_num_points(grid)

27

In [8]:
# evaluate electron repulsion integrals (ERIs) for Gausslet orbitals centered at the grid points,
# taking translational, octahedral and permutational symmetries into account
eri = gli.compute_sparse_eri_gausslet_integrals(b_list, grid, tol)
type(eri)

gausslectronitegrate.SparseERI

In [9]:
# number of stored ERI values after taking symmetries into account
eri.num_entries

1716

In [10]:
# query the integral value for all Gausslets centered at the origin (should agree with above value)
eri.integral_value(([0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]))

1.6252009000830485

In [11]:
# query the integral value at the four other points from above
eri.integral_value((point_a, point_b, point_c, point_d))

-0.0005864045861050586

In [12]:
# as an alternative, load precomputed ERI data from a file
eri_precomp = gli.load_sparse_eri_gausslet_integrals("eri_3x3x3.hdf5")

In [13]:
# precomputed data uses same tolerance
eri_precomp.tol

1e-16

In [14]:
# precomputed data uses same grid specification
eri_precomp.grid

((-1, 3), (-1, 3), (-1, 3))

In [15]:
# precomputed data uses same Gausslet coefficients
import numpy as np
np.linalg.norm(eri_precomp.gausslet_coeffs - b_list)

0.0

In [16]:
# accordingly, integral values agree
eri_precomp.integral_value((point_a, point_b, point_c, point_d)) - eri.integral_value((point_a, point_b, point_c, point_d))

0.0

In [17]:
# project ERI tensor onto a basis
from scipy.stats import ortho_group
rng = np.random.default_rng(293)
# draw 2 random orthogonal state vectors
basis = ortho_group.rvs(gli.grid_num_points(eri.grid), random_state=rng)[:, :2]
eri_tensor_proj = eri.project(basis)
type(eri_tensor_proj)

numpy.ndarray

In [18]:
eri_tensor_proj.shape

(2, 2, 2, 2)

In [19]:
eri_tensor_proj

array([[[[ 0.6573147 , -0.01002096],
         [-0.01002096,  0.63829192]],

        [[-0.01002096,  0.06079474],
         [ 0.06079474, -0.0722164 ]]],


       [[[-0.01002096,  0.06079474],
         [ 0.06079474, -0.0722164 ]],

        [[ 0.63829192, -0.0722164 ],
         [-0.0722164 ,  0.80928236]]]])

In [20]:
# reference calculation
eri_tensor_dense = eri.fill_tensor(grid)
eri_tensor_proj_ref = np.einsum(
    basis, (4, 0),
    basis, (5, 1),
    basis, (6, 2),
    basis, (7, 3),
    eri_tensor_dense, (4, 5, 6, 7),
    (0, 1, 2, 3), optimize=True)
# difference should be zero (up to rounding errors)
np.linalg.norm(eri_tensor_proj - eri_tensor_proj_ref)

2.967646431259338e-14